# Azure Functions

## 1. What is Azure Functions?

**Azure Functions** is a serverless compute service in Azure used to run code in response to events without managing servers.

For an AI application, you can use Functions to implement:

- Agent tools
- REST APIs
- Document-processing pipelines
- Background jobs
- Event-driven workflows
- Scheduled tasks
- Data-processing functions

Simple model:

```text
Event
  ↓
Azure Function
  ↓
Execute Code
  ↓
Result
```

---

# 2. Why Serverless?

With traditional deployment:

```text
VM
 ↓
OS
 ↓
Runtime
 ↓
Application
 ↓
You manage infrastructure
```

With Azure Functions:

```text
Event
 ↓
Azure Functions
 ↓
Your Code
```

Azure manages much of the underlying infrastructure.

You primarily focus on:

```text
Business Logic
```

---

# 3. Azure Functions in Agentic AI ⭐⭐⭐⭐⭐

This is particularly relevant to your Altimetrik JD.

Suppose your agent needs to:

> "Check my leave balance."

Architecture:

```text
User
 ↓
Agent
 ↓
Azure OpenAI
 ↓
Tool Call
 ↓
Azure Function
 ↓
HR Database/API
 ↓
Result
 ↓
Agent
 ↓
User
```

The Function becomes an **agent tool**.

---

# 4. Azure Function as an Agent Tool

Example:

```python
import azure.functions as func

app = func.FunctionApp()

@app.route(
    route="leave-balance",
    methods=["GET"]
)
def leave_balance(req: func.HttpRequest):

    employee_id = req.params.get("employee_id")

    # Call HR database/API
    balance = 12

    return func.HttpResponse(
        f"Leave balance: {balance}"
    )
```

The endpoint could conceptually be:

```text
GET /api/leave-balance?employee_id=123
```

Then your agent can invoke it through a tool/API integration.

---

# 5. Azure Functions + LangGraph

You can make an Azure Function an external tool in your LangGraph workflow.

```text
                    LangGraph
                       │
                       ▼
                    Agent
                       │
                Tool required?
                       │
                       ▼
              Azure Function
                       │
                       ▼
                  HR API/DB
                       │
                       ▼
                  Tool Result
                       │
                       ▼
                    Agent
```

For example:

```python
@tool
def get_leave_balance(employee_id: str):
    """
    Get employee leave balance.
    """

    response = requests.get(
        "https://my-function.azurewebsites.net/api/leave-balance",
        params={"employee_id": employee_id}
    )

    return response.json()
```

Then:

```python
llm_with_tools = llm.bind_tools(
    [get_leave_balance]
)
```

---

# 6. Function Triggers ⭐⭐⭐⭐⭐

Azure Functions can be triggered by different events.

| Trigger | Usage |
|---|---|
| **HTTP** | REST APIs / agent tools |
| **Timer** | Scheduled jobs |
| **Blob Storage** | Process uploaded documents |
| **Queue Storage** | Background processing |
| **Service Bus** | Enterprise messaging |
| **Event Grid** | Event-driven architecture |
| **Event Hubs** | Streaming/event processing |
| **Cosmos DB** | React to database changes |

For AI systems, the most relevant are usually:

```text
HTTP
Blob
Queue
Service Bus
Event Grid
Timer
```

---

# 7. HTTP Trigger

Used when another application calls your Function.

```text
Client
  ↓
HTTP Request
  ↓
Azure Function
  ↓
Response
```

Example:

```python
@app.route(route="ask")
def ask(req: func.HttpRequest):

    question = req.params.get("question")

    return func.HttpResponse(
        f"You asked: {question}"
    )
```

This is useful for:

- AI APIs
- Agent tools
- Backend APIs
- Webhooks

---

# 8. Blob Trigger

Very useful for your **RAG pipeline**.

Suppose a user uploads:

```text
HR_policy.pdf
```

to Blob Storage.

```text
                Blob Storage
                     │
              New PDF uploaded
                     │
                     ▼
               Blob Trigger
                     │
                     ▼
           Azure Document Intelligence
                     │
                     ▼
                  Chunking
                     │
                     ▼
                Embeddings
                     │
                     ▼
             Azure AI Search
```

Example:

```python
@app.blob_trigger(
    arg_name="blob",
    path="documents/{name}",
    connection="AzureWebJobsStorage"
)
def process_document(blob: func.InputStream):

    print(
        f"Processing {blob.name}"
    )

    # Extract document
    # Chunk
    # Generate embeddings
    # Index into Azure AI Search
```

This creates an event-driven document ingestion pipeline.

---

# 9. Queue Trigger

For long-running or asynchronous workloads:

```text
Upload Document
      ↓
Queue Message
      ↓
Azure Function
      ↓
Process Document
```

Example:

```text
User uploads PDF
       ↓
Blob Storage
       ↓
Queue
       ↓
Function
       ↓
Document Intelligence
```

Why use a queue?

Because you don't want the user's HTTP request waiting for a potentially long document-processing operation.

---

# 10. Service Bus Trigger

For enterprise systems, **Azure Service Bus** is commonly used for reliable asynchronous messaging.

```text
Application
     ↓
Service Bus
     ↓
Azure Function
     ↓
AI Processing
```

Example:

```text
Invoice Service
      ↓
Service Bus
      ↓
Invoice Processing Function
      ↓
Document Intelligence
      ↓
AI Extraction
```

This is more enterprise-oriented than simply making synchronous HTTP calls.

---

# 11. Timer Trigger

Useful for scheduled AI jobs.

Example:

```text
Every night at 12 AM
       ↓
Azure Function
       ↓
Process new documents
       ↓
Evaluate RAG
       ↓
Generate report
```

Python concept:

```python
@app.timer_trigger(
    schedule="0 0 0 * * *",
    arg_name="timer"
)
def nightly_job(timer):
    print("Running nightly job")
```

---

# 12. Event-Driven RAG Architecture

A production ingestion flow could be:

```text
                     User
                       │
                       ▼
                 Upload PDF
                       │
                       ▼
                Azure Blob Storage
                       │
                 Blob Event
                       │
                       ▼
               Azure Function
                       │
                       ▼
            Document Intelligence
                       │
                       ▼
                  Chunking
                       │
                       ▼
                 Embeddings
                       │
                       ▼
              Azure AI Search
                       │
                       ▼
                 RAG Ready
```

This is much better than having your web application synchronously process a 200-page PDF.

---

# 13. Azure Functions + Azure OpenAI

A Function can also expose an AI endpoint.

```text
Client
 ↓
API Management
 ↓
Azure Function
 ↓
Azure OpenAI
 ↓
Response
```

Example:

```python
@app.route(route="summarize")
def summarize(req):

    text = req.get_json()["text"]

    # Call Azure OpenAI

    return func.HttpResponse(
        "Summary"
    )
```

---

# 14. Azure Functions + Managed Identity

This is important.

Don't do:

```python
search_key = "xxxxxxxx"
```

Instead:

```text
Azure Function
      │
Managed Identity
      │
Microsoft Entra ID
      │
      ├── Azure AI Search
      ├── Blob Storage
      ├── Key Vault
      └── Azure OpenAI
```

Python:

```python
from azure.identity import DefaultAzureCredential

credential = DefaultAzureCredential()
```

Then pass the credential to Azure SDK clients where supported.

---

# 15. Azure Functions + Key Vault

If you genuinely need a secret:

```text
Azure Function
      ↓
Managed Identity
      ↓
Entra ID
      ↓
Key Vault
      ↓
Third-party API Key
```

This keeps secrets outside your source code.

---

# 16. Azure Functions + API Management

For enterprise AI APIs, a common architecture is:

```text
Client
  ↓
Azure API Management
  ↓
Azure Function
  ↓
Azure OpenAI / Agent / RAG
```

API Management can provide capabilities such as:

- Authentication/authorization integration
- Rate limiting
- Quotas
- API policies
- Monitoring
- API versioning

Example:

```text
POST /api/hr-agent
       ↓
API Management
       ↓
Azure Function
       ↓
LangGraph Agent
       ↓
Azure OpenAI
```

---

# 17. Azure Functions + Content Safety

You can put safety controls inside the AI API:

```text
User
 ↓
API Management
 ↓
Azure Function
 ↓
Content Safety
 ↓
Agent / Azure OpenAI
 ↓
Content Safety
 ↓
Response
```

Example:

```python
def ai_endpoint(question):

    # 1. Input safety
    if not is_safe(question):
        return "Request blocked."

    # 2. Agent / LLM
    answer = run_agent(question)

    # 3. Output safety
    if not is_safe(answer):
        return "Response blocked."

    return answer
```

---

# 18. Azure Functions vs Container/VM

| Azure Functions | VM / Container |
|---|---|
| Serverless | Infrastructure/application runtime management |
| Event-driven | Flexible long-running workloads |
| Easy scaling | More control |
| Good for APIs/events | Good for complex applications |
| Good for short/medium tasks | Better for long-running workloads |
| Pay based on selected hosting/usage model | Pay for provisioned infrastructure |
| Less infrastructure management | More operational control |

Don't say:

> "Functions are always cheaper."

Cost depends on the workload, hosting plan, execution duration, scale, networking, and other factors.

---

# 19. Important Limitation

Functions aren't automatically the best choice for every AI workload.

For example:

```text
200-page document
       ↓
Complex AI processing
       ↓
30-minute execution
```

You should consider:

- Durable Functions
- Queue-based architecture
- Container Apps
- AKS
- Batch processing

depending on the workload.

---

# 20. Durable Functions ⭐⭐⭐⭐

**Durable Functions** extends Azure Functions for stateful serverless workflows.

This is useful for multi-step AI workflows.

Example:

```text
Start
 ↓
Extract Document
 ↓
Chunk
 ↓
Generate Embeddings
 ↓
Index
 ↓
Evaluate
 ↓
Complete
```

Durable Functions can orchestrate these steps.

Conceptually:

```text
                 Orchestrator
                     │
        ┌────────────┼────────────┐
        ▼            ▼            ▼
    Activity A   Activity B   Activity C
        │            │            │
        └────────────┼────────────┘
                     ▼
                   END
```

---

# 21. Azure Functions vs LangGraph

They solve different problems.

| Azure Functions | LangGraph |
|---|---|
| Compute/runtime | Agent orchestration |
| Event triggers | Nodes/edges/state |
| Serverless execution | Agent workflow |
| HTTP/API | Tool orchestration |
| Blob/Queue triggers | Conditional routing |
| Durable Functions for workflows | Checkpointing/state |

You can use them together:

```text
LangGraph Agent
      │
      ▼
Azure Function
      │
      ▼
Business API
```

---

# 22. Azure Functions + Agentic AI

A production architecture:

```text
                           User
                             │
                             ▼
                    Teams / Web / API
                             │
                             ▼
                     API Management
                             │
                             ▼
                      Azure Function
                             │
                             ▼
                      LangGraph Agent
                             │
            ┌────────────────┼────────────────┐
            ▼                ▼                ▼
       Azure OpenAI    Azure AI Search    Functions
            │                                 │
            │                           Business APIs
            │
            ▼
       Content Safety
                             │
                             ▼
                          Response
```

Supporting:

```text
Entra ID
Managed Identity
Key Vault
Azure Monitor
Application Insights
```

---

# 23. Production Considerations

For an AI Function, consider:

### Security

- Entra ID
- Managed Identity
- RBAC
- API Management
- Key Vault
- Network restrictions

### Reliability

- Retries
- Timeouts
- Dead-letter queues
- Idempotency

### AI-specific

- Token limits
- LLM timeout
- Rate limits
- Prompt injection
- Content Safety
- Tool authorization

### Observability

- Application Insights
- Logs
- Metrics
- Distributed tracing
- LLM/tool latency

---

# 24. Common Interview Questions

### Q1. What is Azure Functions?

> "Azure Functions is a serverless, event-driven compute service that allows us to execute application code without managing the underlying server infrastructure."

### Q2. How would you use Azure Functions in an AI application?

> "I can use HTTP-triggered Functions as AI APIs or agent tools, Blob-triggered Functions for document ingestion, Queue or Service Bus triggers for asynchronous processing, and Timer triggers for scheduled AI jobs."

### Q3. How would you integrate Functions with an Agent?

> "I would expose the required business capability as an authenticated API or function, define it as a tool for the agent, and enforce authorization and input validation before executing the underlying business operation."

### Q4. Why use a queue with Functions?

> "For asynchronous and decoupled processing. For example, after a PDF upload, I can publish a message and let a Function process the document independently instead of keeping the user's HTTP request open."

### Q5. How do you secure an Azure Function?

> "I would use Entra ID for authentication, Managed Identity for downstream Azure resources, RBAC for authorization, API Management for API-level controls, Key Vault for secrets that cannot be eliminated, and appropriate network restrictions."

### Q6. Azure Function vs LangGraph?

> "Functions provide compute and event-driven execution, whereas LangGraph provides agent workflow orchestration. They can complement each other—for example, LangGraph can orchestrate an agent while an Azure Function implements one of its business tools."

---

# 25. Senior-Level Scenario

### Interviewer:

> "A user uploads a 200-page PDF. Design the Azure architecture."

A strong answer:

```text
User
 ↓
Blob Storage
 ↓
Blob Event
 ↓
Queue / Service Bus
 ↓
Azure Function
 ↓
Document Intelligence
 ↓
Chunk + Metadata
 ↓
Embeddings
 ↓
Azure AI Search
 ↓
RAG Ready
```

Then:

> "I would make ingestion asynchronous rather than processing the 200-page document synchronously through an HTTP request. Blob Storage would hold the original document, an event or queue would trigger processing, Azure Functions would orchestrate individual processing steps, Document Intelligence would extract structured content, and the resulting chunks would be embedded and indexed into Azure AI Search. I would use Managed Identity for Azure-to-Azure authentication and Application Insights for monitoring."

That connects **Azure Functions + Document Intelligence + RAG + AI Search + Managed Identity + production architecture** into one coherent answer.